# Create Training Labels from the Critical Mineral Deposits Database
from shapefile to GeoTIFF


In [12]:
import numpy as np
import geopandas as gpd

import sys
if sys.version_info < (3, 9):
    from importlib_resources import files
else:
    from importlib.resources import files

from beak.experimental.io import save_raster, load_raster

# Load data

**User definitions**

In [13]:
BASE_PATH = files("beak.data")
PATH_SHAPEFILE = BASE_PATH / "CRITICAL_MINERAL_DEPOSITS" / "critical_mineral_deposits.shp"
PATH_BASE_RASTER = BASE_PATH / "BASE_RASTERS" / "EPSG_4326_RES_0_025_US_CONT_ALASKA.tif"


# Create Labels
## Complete extent of the dataset

In [5]:
from beak.experimental.conversions import create_binary_raster

gdf = gpd.read_file(PATH_SHAPEFILE)
base_raster = load_raster(PATH_BASE_RASTER)
out_path = BASE_PATH / "TRAINING_LABELS" / "MAGMATIC_NICKEL_NAT" / "MAGMATIC_NICKEL_EPSG_4326_RES_0_025_US_CONT_ALASKA.tif"

# Create query to select relevant mineral occurences
SQL_QUERY = "DepType == 'Nickel-copper-PGE sulfide'"

out_array = create_binary_raster(gdf, base_raster, query=SQL_QUERY, all_touched=False, same_shape=True, fill_negatives=True, out_file=None)

# Check results
np.unique(out_array, return_counts=True)

(array([-99,   0,   1], dtype=int8),
 array([31856329,  1782097,       25], dtype=int64))

## Clip to conterminous U.S. (lower 48 states)

In [10]:
from beak.experimental.raster_processing import unify_raster_grids

PATH_BASE_RASTER_US_CONT = load_raster(BASE_PATH / "BASE_RASTERS" / "EPSG_4326_RES_0_025_US_CONT.tif")
unified_raster, meta = unify_raster_grids(base_raster, [out_path], same_extent=True, same_shape=True)[0]

np.unique(unified_raster, return_counts=True)

(array([-99.,   0.,   1.]), array([31856329,  1782097,       25], dtype=int64))

In [ ]:
# Save the unified/clipped raster
out_path = BASE_PATH / "TRAINING_LABELS" / "MAGMATIC_NICKEL_NAT" / "MAGMATIC_NICKEL_EPSG_4326_RES_0_025_US_CONT.tif"

save = False
if save is True:
  save_raster(out_path, array=unified_raster, dtype=np.dtype(np.int8), metadata=meta)
  